Below is a complete Google Colab workflow for your approximately 1,800 Airbus S.A.S. EASA AD PDFs.

The process is read-only for your original PDFs. It creates:

* A master corpus manifest
* Exact binary duplicate groups
* Exact text duplicate groups
* Same-AD conflicting-file warnings
* Revision and correction chains
* Possible supersedure relationships
* Near-duplicate candidates
* OCR and metadata quality-control reports

EASA’s official convention is important:

* Normal AD: `2023-0123`
* Revision: `2023-0123R1`, `2023-0123R2`
* Emergency AD: `2023-0123-E`
* Correction: usually retains the AD number and contains `[Corrected: date]`

EASA states that `R1`, `R2`, etc. are appended without a space, while corrections display their publication date in brackets. [[EASA AD Writing Instructions](https://www.easa.europa.eu/en/downloads/44083/en)](https://www.easa.europa.eu/en/downloads/44083/en).

# 1. Prepare your Google Drive folders

Create this structure:


```text
My Drive/
└── Capstone_AD_Project/
    ├── corpus_raw/
    │   ├── AD_1.pdf
    │   ├── AD_2.pdf
    │   └── ...
    └── metadata/
```

You can use subfolders inside `corpus_raw`. The program searches recursively.

Important:

* Keep `corpus_raw` unchanged.
* Do not delete or rename duplicates yet.
* Do not place generated reports inside `corpus_raw`.
* Make sure the folder contains only the Airbus S.A.S. AD corpus you want to study.

# 2. Create a Google Colab notebook

Open [[Google Colab](https://colab.research.google.com/)](https://colab.research.google.com/) and create a new notebook named:


```text
01_build_ad_corpus_manifest.ipynb
```

Select:


```text
Runtime → Change runtime type → Python 3
```

A GPU is not required for this step.

# 3. Install the required libraries

Run this as the first cell:


In [ ]:
!pip install -q \
    pymupdf \
    pandas \
    pyarrow \
    openpyxl \
    python-dateutil \
    scikit-learn \
    tqdm


These libraries are used for:

* `PyMuPDF`: extracting PDF text and metadata
* `pandas`: constructing the manifest
* `pyarrow`: saving compressed Parquet files
* `openpyxl`: creating an Excel version of the manifest
* `scikit-learn`: near-duplicate detection
* `tqdm`: progress display

# 4. Mount Google Drive and configure the paths

Run:


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Then configure the folders:


In [ ]:
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Capstone_AD_Project"
)

RAW_DIR = PROJECT_DIR / "corpus_raw"
OUTPUT_DIR = PROJECT_DIR / "metadata"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert RAW_DIR.exists(), f"Folder not found: {RAW_DIR}"

pdf_paths = sorted(RAW_DIR.rglob("*.pdf"))

print("Raw PDF folder:", RAW_DIR)
print("Output folder:", OUTPUT_DIR)
print("Number of PDF files found:", len(pdf_paths))

assert len(pdf_paths) > 0, "No PDF files were found."


The result should be close to:


```text
Number of PDF files found: 1800
```

If the result is zero, check the path and capitalization.

# 5. Define the manifest fields

The program will create the following main fields:

| Field                    | Purpose                                                 |
| ------------------------ | ------------------------------------------------------- |
| `file_instance_id`       | Unique identifier for this physical file/path           |
| `content_id`             | Identifier based on file contents                       |
| `file_name`              | Original PDF filename                                   |
| `relative_path`          | Location under `corpus_raw`                             |
| `file_size_bytes`        | PDF size                                                |
| `page_count`             | Number of pages                                         |
| `extracted_char_count`   | Native text extraction result                           |
| `needs_ocr`              | Whether text extraction appears insufficient            |
| `file_sha256`            | Detects byte-for-byte duplicates                        |
| `normalized_text_sha256` | Detects PDFs with identical text but different metadata |
| `ad_number`              | Full AD number, such as `2022-0123R2`                   |
| `base_ad_number`         | Original family number, such as `2022-0123`             |
| `revision_number`        | `0`, `1`, `2`, etc.                                     |
| `is_emergency`           | Whether the number ends in `-E`                         |
| `is_correction`          | Whether the document contains a correction notice       |
| `correction_date`        | Date found in `[Corrected: ...]`                        |
| `issue_date`             | Extracted AD issue date                                 |
| `supersedes_ad_numbers`  | Older ADs that this AD may supersede                    |
| `duplicate_type`         | Binary, text or same-version conflict                   |
| `duplicate_of`           | Master file for an exact duplicate                      |
| `previous_version`       | Previous item in the revision chain                     |
| `next_version`           | Next item in the revision chain                         |
| `is_latest_version`      | Whether this is the latest available version            |
| `review_flags`           | Problems needing manual review                          |

# 6. Add the helper functions

Run this complete cell:


In [ ]:
import fitz
import hashlib
import json
import re
import unicodedata

import pandas as pd

from collections import defaultdict
from datetime import datetime, timezone
from dateutil import parser as date_parser
from tqdm.auto import tqdm


Then run the following functions:


In [ ]:
# ---------------------------------------------------------
# Hashing
# ---------------------------------------------------------

def sha256_file(path, block_size=1024 * 1024):
    """Calculate a SHA-256 hash without loading the whole PDF into memory."""
    hasher = hashlib.sha256()

    with open(path, "rb") as file:
        while True:
            block = file.read(block_size)

            if not block:
                break

            hasher.update(block)

    return hasher.hexdigest()


def stable_path_id(relative_path):
    """Unique ID for a physical file path."""
    return hashlib.sha1(
        str(relative_path).encode("utf-8")
    ).hexdigest()[:16]


# ---------------------------------------------------------
# PDF extraction
# ---------------------------------------------------------

def extract_pdf_text(path):
    """
    Extract native text from a PDF.

    This does not perform OCR. Poorly extracted files are flagged
    for OCR in a later stage.
    """
    pages = []
    metadata = {}

    with fitz.open(path) as document:
        page_count = document.page_count
        metadata = document.metadata or {}

        for page in document:
            page_text = page.get_text("text", sort=True)
            pages.append(page_text)

    full_text = "\n".join(pages)

    return {
        "text": full_text,
        "pages": pages,
        "page_count": page_count,
        "pdf_metadata": metadata,
    }


# ---------------------------------------------------------
# Text normalization
# ---------------------------------------------------------

def normalize_text(text):
    """
    Normalize text for duplicate detection.

    This version:
    - standardizes Unicode;
    - removes soft hyphens;
    - joins words divided by line-break hyphenation;
    - lowercases text;
    - collapses repeated whitespace.
    """
    if not text:
        return ""

    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\u00ad", "")

    # Example:
    # "airworthi-\nness" -> "airworthiness"
    text = re.sub(
        r"(?<=\w)-\s*\n\s*(?=\w)",
        "",
        text,
    )

    text = text.lower()
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def sha256_text(text):
    if not text:
        return ""

    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


# 7. Define EASA AD-number parsing

Run:


In [ ]:
AD_HEADER_RE = re.compile(
    r"""
    ^[ \t]*
    (?:EASA\s+)?
    (?:EMERGENCY\s+)?
    AD\s+No\.?
    \s*[:#]?\s*
    (?P<year>(?:19|20)\d{2})
    \s*[-–—_]\s*
    (?P<number>\d{4})
    (?:\s*[-_]?\s*(?P<revision>R\d+))?
    (?:\s*[-_]?\s*(?P<emergency>E))?
    \b
    """,
    re.IGNORECASE | re.MULTILINE | re.VERBOSE,
)


GENERIC_AD_RE = re.compile(
    r"""
    \b
    (?P<year>(?:19|20)\d{2})
    [-–—_]
    (?P<number>\d{4})
    (?:[-_\s]?(?P<revision>R\d+))?
    (?:[-_\s]?(?P<emergency>E))?
    \b
    """,
    re.IGNORECASE | re.VERBOSE,
)


FILENAME_AD_RE = re.compile(
    r"""
    (?P<year>(?:19|20)\d{2})
    [-–—_]
    (?P<number>\d{4})
    (?:[-_\s]?(?P<revision>R\d+))?
    (?:[-_\s]?(?P<emergency>E))?
    """,
    re.IGNORECASE | re.VERBOSE,
)


def build_ad_number(match):
    """Create a standardized EASA AD number from a regex match."""
    year = match.group("year")
    number = match.group("number")

    revision = match.groupdict().get("revision")
    emergency = match.groupdict().get("emergency")

    ad_number = f"{year}-{number}"

    if revision:
        ad_number += revision.upper()

    if emergency and not revision:
        ad_number += "-E"

    return ad_number


def parse_ad_components(ad_number):
    """
    Convert 2023-0123R2 into:
    base_ad_number = 2023-0123
    revision_number = 2
    is_emergency = False
    """
    if not ad_number:
        return "", 0, False

    match = re.fullmatch(
        r"((?:19|20)\d{2}-\d{4})(?:R(\d+))?(-E)?",
        ad_number,
        flags=re.IGNORECASE,
    )

    if not match:
        return "", 0, False

    base_number = match.group(1).upper()
    revision_number = int(match.group(2) or 0)
    is_emergency = bool(match.group(3))

    return base_number, revision_number, is_emergency


def find_ad_number(text, file_name):
    """
    Search order:

    1. Anchored first-page ``AD No.`` line
    2. PDF filename
    3. Generic first-page match

    The header and filename are parsed independently so disagreements can
    be surfaced instead of silently accepting a historical body reference.
    """
    first_part = text[:5000] if text else ""

    header_boundary = re.search(
        r"^[ \t]*(?:Issued|Date)[ \t]*:",
        first_part,
        flags=re.IGNORECASE | re.MULTILINE,
    )
    header_region = (
        first_part[:header_boundary.start()]
        if header_boundary
        else first_part
    )

    header_match = AD_HEADER_RE.search(header_region)
    filename_match = FILENAME_AD_RE.search(Path(file_name).stem)

    header_number = (
        build_ad_number(header_match)
        if header_match
        else ""
    )
    filename_number = (
        build_ad_number(filename_match)
        if filename_match
        else ""
    )

    if header_number:
        if filename_number and header_number != filename_number:
            return (
                header_number,
                "pdf_header_filename_mismatch",
                0.75,
            )

        return header_number, "pdf_header", 1.00

    if filename_number:
        return filename_number, "filename", 0.90

    generic_match = GENERIC_AD_RE.search(first_part)

    if generic_match:
        return build_ad_number(generic_match), "generic_first_page", 0.60

    return "", "not_found", 0.00


AD_NUMBER_REGRESSION_CASES = [
    (
        "EASA AIRWORTHINESS DIRECTIVE\n"
        "AD No.: 2007 - 0281\n"
        "Supersedure: This AD supersedes EASA AD 2006-0047",
        "2007-0281__easa_ad_2007_0281.pdf",
        "2007-0281",
        "pdf_header",
    ),
    (
        "EASA AIRWORTHINESS DIRECTIVE\n"
        "AD No.: 2008 – 0032\n"
        "Supersedure: EASA AD 2006-0108",
        "2008-0032__easa_ad_2008_0032.pdf",
        "2008-0032",
        "pdf_header",
    ),
    (
        "EASA AD No.: 2022-0096R2\nIssued: 12 April 2024",
        "2022-0096R2__EASA_AD_2022_0096_R2.pdf",
        "2022-0096R2",
        "pdf_header",
    ),
    (
        "Reason: SB A320-27-1164 was mandated by EASA AD 2006-0223.",
        "2007-0178__easa_ad_2007_0178.pdf",
        "2007-0178",
        "filename",
    ),
    (
        "Issued: 20 June 2007\n"
        "Supersedure:\n"
        "EASA AD No. 2006-0223 is superseded by this AD.",
        "2007-0178__easa_ad_2007_0178.pdf",
        "2007-0178",
        "filename",
    ),
]


for (
    regression_text,
    regression_file_name,
    expected_ad_number,
    expected_source,
) in AD_NUMBER_REGRESSION_CASES:
    parsed_number, parsed_source, _ = find_ad_number(
        regression_text,
        regression_file_name,
    )
    assert parsed_number == expected_ad_number
    assert parsed_source == expected_source


mismatch_number, mismatch_source, mismatch_confidence = find_ad_number(
    "EASA AD No.: 2024-0091R1\nIssued: 30 May 2024",
    "2024-0092__incorrect_filename.pdf",
)
assert mismatch_number == "2024-0091R1"
assert mismatch_source == "pdf_header_filename_mismatch"
assert mismatch_confidence == 0.75

print("AD number parser validation passed.")


The parser reads an anchored `AD No.` line from the first page and allows spaces around legacy separators such as `2007 - 0281` and `2008 – 0032`.

It separately parses the filename. If the two identifiers disagree, the cover-page header is retained but the row is flagged for manual review instead of silently grouping it with another AD.

Only when no anchored header is available does the parser use the filename, followed by a low-confidence generic fallback that is explicitly flagged for manual review.

# 8. Define date, correction and relationship extraction

Run:


In [ ]:
CORRECTION_RE = re.compile(
    r"\[\s*Corrected\s*:\s*([^\]]+)\]",
    re.IGNORECASE,
)


ISSUE_DATE_PATTERNS = [
    re.compile(
        r"\bIssue\s+date\s*:\s*"
        r"([0-3]?\d\s+[A-Za-z]+\s+(?:19|20)\d{2})",
        re.IGNORECASE,
    ),
    re.compile(
        r"\bDate\s*:\s*"
        r"([0-3]?\d\s+[A-Za-z]+\s+(?:19|20)\d{2})",
        re.IGNORECASE,
    ),
]


def parse_date_safely(raw_date):
    if not raw_date:
        return ""

    try:
        parsed = date_parser.parse(
            raw_date,
            dayfirst=True,
            fuzzy=True,
        )

        return parsed.date().isoformat()

    except Exception:
        return ""


def extract_correction_information(text):
    match = CORRECTION_RE.search(text or "")

    if not match:
        return False, "", ""

    raw_date = match.group(1).strip()
    parsed_date = parse_date_safely(raw_date)

    return True, raw_date, parsed_date


def extract_issue_date(text):
    first_part = text[:20000] if text else ""

    for pattern in ISSUE_DATE_PATTERNS:
        match = pattern.search(first_part)

        if match:
            raw_date = match.group(1).strip()
            parsed_date = parse_date_safely(raw_date)

            return raw_date, parsed_date

    return "", ""


Add supersedure detection:


In [ ]:
SUPERSEDURE_FIELD_LABEL_RE = re.compile(
    r"\b(?:Revision\s*/\s*)?Supersedure\s*:\s*",
    re.IGNORECASE,
)


SUPERSEDURE_FIELD_BOUNDARY_RE = re.compile(
    r"^(?:"
    r"ATA\s*\d{2}\b|"
    r"Manufacturer(?:\(s\))?\s*:|"
    r"Applicability\s*:|"
    r"Effective\s+Date\s*:|"
    r"Required\s+Action|"
    r"Compliance\s*:|"
    r"Reason\s*:|"
    r"TCDS\s+Number|"
    r"Foreign\s+AD\s*:|"
    r"(?:Revision\s*/\s*)?Supersedure\s*:"
    r")",
    re.IGNORECASE,
)


NO_SUPERSEDURE_RE = re.compile(
    r"^(?:none|not\s+applicable|n\s*/\s*a)\b",
    re.IGNORECASE,
)


POSITIVE_SUPERSEDURE_RE = re.compile(
    r"\b(?:"
    r"supersedes(?:\s+and\s+cancels)?|"
    r"superseded|"
    r"cancels\s+and\s+(?:replaces|supersedes)"
    r")\b",
    re.IGNORECASE,
)


REVERSE_OR_NEGATED_SUPERSEDURE_RE = re.compile(
    r"\b(?:"
    r"(?:this|the\s+present)\s+(?:EASA\s+)?AD\s+"
    r"(?:is|was)\s+superseded\s+by|"
    r"(?:does|is|are|was|were)\s+not\s+"
    r"(?:supersede|superseded)"
    r")\b",
    re.IGNORECASE,
)


EXPLICIT_FORWARD_SUPERSEDURE_RE = re.compile(
    r"\b(?:"
    r"(?:this|the\s+present|the\s+original)\s+"
    r"(?:new\s+)?(?:EASA\s+)?"
    r"(?:airworthiness\s+directive\s*(?:\(AD\))?|AD)|"
    r"the\s+original\s+issue\s+of\s+this\s+(?:EASA\s+)?AD"
    r")\s*,?\s*"
    r"(?:(?:which|also|hereby)\s+){0,2}"
    r"(?:"
    r"supersedes(?:\s+and\s+cancels)?|"
    r"cancels\s+and\s+(?:replaces|supersedes)"
    r")\s+"
    r"(?P<targets>[^.\n]{1,500})",
    re.IGNORECASE,
)


EXPLICIT_PASSIVE_SUPERSEDURE_RE = re.compile(
    # Begin at an AD marker instead of trying every character in the PDF.
    # The previous leading wildcard was bounded but still caused heavy
    # backtracking across the 1,809-document corpus.
    r"\b(?P<targets>(?:"
    r"(?:EASA\s+)?(?:Emergency\s+)?E?ADs?|"
    r"Airworthiness\s+Directives?"
    r")\b[^.\n]{1,300}?)\s+"
    r"(?:is|are|was|were)\s+(?:therefore\s+)?"
    r"superseded\s+by\s+"
    r"(?:this|the\s+present)\s+(?:EASA\s+)?AD\b",
    re.IGNORECASE,
)


EXPLICIT_RETAINS_SUPERSEDED_RE = re.compile(
    r"\b(?:this|the\s+present)\s+"
    r"(?:new\s+)?(?:EASA\s+)?AD\s+"
    r"retains\s+(?:the\s+)?requirements?\s+of\s+"
    r"(?P<targets>[^.\n]{1,500}?)\s*,?\s*"
    r"which\s+(?:is|are|was|were)\s+superseded\b",
    re.IGNORECASE,
)


AD_REFERENCE_MARKER_RE = re.compile(
    r"(?:"
    r"\b(?:EASA\s+)?(?:Emergency\s+)?E?ADs?\b|"
    r"\bAirworthiness\s+Directive\b"
    r")",
    re.IGNORECASE,
)


NON_AD_NUMBER_CONTEXT_RE = re.compile(
    r"(?:"
    r"approval(?:\s+number)?|"
    r"service\s+bulletin|"
    r"\bSB|"
    r"TCDS(?:\s+number)?"
    r")\s*(?:No\.?\s*)?$",
    re.IGNORECASE,
)


def extract_supersedure_field_values(text):
    """Read the authoritative EASA Supersedure header field."""
    lines = (text or "").splitlines()
    values = []

    for line_index, line in enumerate(lines):
        label_match = SUPERSEDURE_FIELD_LABEL_RE.search(line)

        if not label_match:
            continue

        value_parts = [line[label_match.end():].strip()]

        if NO_SUPERSEDURE_RE.match(value_parts[0]):
            values.append(value_parts[0])
            continue

        # A long field can wrap over a few PDF text lines. Stop before the
        # next structured header and never scan the body as part of the field.
        for continuation in lines[line_index + 1:line_index + 5]:
            continuation = continuation.strip()

            if not continuation:
                break

            if SUPERSEDURE_FIELD_BOUNDARY_RE.match(continuation):
                break

            if value_parts[-1].rstrip().endswith((".", ";")):
                break

            value_parts.append(continuation)

        values.append(" ".join(part for part in value_parts if part).strip())

    return values


def extract_referenced_ad_numbers(text, current_base):
    """Return AD-number references, excluding dates and approval numbers."""
    references = set()

    for ad_match in GENERIC_AD_RE.finditer(text or ""):
        full_prefix = (text or "")[:ad_match.start()]
        near_prefix = full_prefix[-100:]

        if NON_AD_NUMBER_CONTEXT_RE.search(near_prefix):
            continue

        # A plural marker can introduce a long list only once, for example
        # ``EASA ADs 2007-0300, 2008-0152 and 2009-0191``. Require that marker
        # before the number inside this already-bounded relation, while the
        # near-prefix check above still rejects approval and SB numbers.
        if not AD_REFERENCE_MARKER_RE.search(full_prefix):
            continue

        candidate = build_ad_number(ad_match)
        candidate_base, _, _ = parse_ad_components(candidate)

        if candidate_base and candidate_base != current_base:
            references.add(candidate)

    return references


def extract_supersedure_candidates(text, current_ad_number):
    """
    Extract high-precision supersedure candidates.

    Priority 1 is the structured EASA ``Supersedure:`` field. Strongly
    directional sentences whose subject is the current AD are also accepted
    because PDF extraction can split or reorder that field. Broad keyword
    proximity is intentionally not used.
    """
    current_base, _, _ = parse_ad_components(current_ad_number)
    candidates = set()
    evidence = set()

    field_values = extract_supersedure_field_values(text)

    if field_values:
        for field_value in field_values:
            normalized_value = re.sub(r"\s+", " ", field_value).strip()

            if not normalized_value:
                continue

            if NO_SUPERSEDURE_RE.match(normalized_value):
                continue

            if REVERSE_OR_NEGATED_SUPERSEDURE_RE.search(normalized_value):
                continue

            if not POSITIVE_SUPERSEDURE_RE.search(normalized_value):
                continue

            found = extract_referenced_ad_numbers(
                normalized_value,
                current_base,
            )

            if found:
                candidates.update(found)
                evidence.add(f"Supersedure: {normalized_value}"[:600])

    # PDF text layout can split or reorder a structured field. Independently
    # accept strongly directional sentences whose grammatical subject is the
    # current AD. This restores recall without returning to keyword proximity.
    relation_source = re.sub(r"\s+", " ", text or "")
    fallback_matches = [
        *EXPLICIT_FORWARD_SUPERSEDURE_RE.finditer(relation_source),
        *EXPLICIT_PASSIVE_SUPERSEDURE_RE.finditer(relation_source),
        *EXPLICIT_RETAINS_SUPERSEDED_RE.finditer(relation_source),
    ]

    if current_base:
        current_family_subject_re = re.compile(
            rf"\b(?:EASA\s+)?AD\s+{re.escape(current_base)}"
            r"(?:R\d+)?\s+(?:(?:also|hereby)\s+)?"
            r"superseded\s+(?P<targets>[^.\n]{1,500})",
            re.IGNORECASE,
        )
        fallback_matches.extend(
            current_family_subject_re.finditer(relation_source)
        )

    for relation_match in fallback_matches:
        relation_text = relation_match.group(0)

        if REVERSE_OR_NEGATED_SUPERSEDURE_RE.search(relation_text):
            continue

        found = extract_referenced_ad_numbers(
            relation_match.group("targets"),
            current_base,
        )

        if found:
            candidates.update(found)
            evidence.add(re.sub(r"\s+", " ", relation_text).strip()[:600])

    return sorted(candidates), sorted(evidence)


Why these remain candidate relationships:

* The structured EASA `Supersedure:` field is authoritative, including when it says `None`.
* Explicit directional statements such as `This AD supersedes EASA AD ...` are also accepted because PDF text extraction can split or reorder the structured field.
* Negated statements, reverse `SUPERSEDED BY` stamps, approval numbers and broad keyword proximity are ignored.
* Automated extraction can still be affected by unusual PDF text layout, so exported links retain `manually_verified=False` until reviewed.

# 9. Scan all PDFs and construct initial records

Run:


In [ ]:
# Fast reruns reuse the prior manifest and extracted-text cache.
# Set this to True only when the source PDFs themselves have changed.
FORCE_RESCAN_PDFS = False

cached_manifest_path = OUTPUT_DIR / "corpus_manifest.parquet"
cached_text_path = OUTPUT_DIR / "corpus_extracted_text.parquet"
use_cached_corpus = (
    not FORCE_RESCAN_PDFS
    and cached_manifest_path.exists()
    and cached_text_path.exists()
)

if not FORCE_RESCAN_PDFS:
    missing_cache_paths = [
        str(path)
        for path in (cached_manifest_path, cached_text_path)
        if not path.exists()
    ]
    assert not missing_cache_paths, (
        "Cached rerun requested, but cache files are missing: "
        + ", ".join(missing_cache_paths)
        + ". Set FORCE_RESCAN_PDFS = True only if a full PDF rescan "
        "is intentional."
    )

if use_cached_corpus:
    cached_manifest = pd.read_parquet(cached_manifest_path)
    cached_text_df = pd.read_parquet(cached_text_path)

    assert len(cached_manifest) == len(pdf_paths)
    assert cached_manifest["file_instance_id"].is_unique
    assert cached_text_df["file_instance_id"].is_unique
    assert set(cached_manifest["file_instance_id"]) == set(
        cached_text_df["file_instance_id"]
    )

    records = cached_manifest.to_dict("records")
    text_cache = cached_text_df.to_dict("records")
    text_by_file_id = dict(zip(
        cached_text_df["file_instance_id"],
        cached_text_df["text"],
    ))
    normalized_text_by_file_id = {
        file_instance_id: normalize_text(text)
        for file_instance_id, text in text_by_file_id.items()
    }

    list_columns = [
        "supersedes_ad_numbers",
        "superseded_by_ad_numbers",
        "supersedure_evidence",
        "review_flags",
    ]

    for record in tqdm(records, desc="Refreshing cached parser and relationship fields"):
        for column in list_columns:
            value = record.get(column, "")

            if isinstance(value, list):
                parsed_value = value
            elif pd.isna(value) or value == "":
                parsed_value = []
            else:
                parsed_value = [
                    part.strip()
                    for part in str(value).split(" | ")
                    if part.strip()
                ]

            record[column] = parsed_value

        raw_text = text_by_file_id.get(record["file_instance_id"], "")
        (
            ad_number,
            ad_number_source,
            ad_number_confidence,
        ) = find_ad_number(
            raw_text,
            record.get("file_name", ""),
        )
        (
            base_ad_number,
            revision_number,
            is_emergency,
        ) = parse_ad_components(ad_number)
        (
            is_correction,
            correction_date_raw,
            correction_date,
        ) = extract_correction_information(raw_text)
        issue_date_raw, issue_date = extract_issue_date(raw_text)

        record["ad_number"] = ad_number
        record["base_ad_number"] = base_ad_number
        record["revision_number"] = revision_number
        record["is_emergency"] = is_emergency
        record["is_correction"] = is_correction
        record["correction_date_raw"] = correction_date_raw
        record["correction_date"] = correction_date
        record["issue_date_raw"] = issue_date_raw
        record["issue_date"] = issue_date
        record["ad_number_source"] = ad_number_source
        record["ad_number_confidence"] = ad_number_confidence

        supersedes_ad_numbers, supersedure_evidence = (
            extract_supersedure_candidates(
                raw_text,
                ad_number,
            )
        )
        record["supersedes_ad_numbers"] = supersedes_ad_numbers
        record["supersedure_evidence"] = supersedure_evidence

        is_airbus = bool(re.search(
            r"\bAirbus\b",
            raw_text,
            flags=re.IGNORECASE,
        ))
        record["is_airbus_sas_detected"] = is_airbus
        recomputed_flags = {
            "ad_number_not_found",
            "ad_number_header_filename_mismatch",
            "generic_ad_number_requires_review",
            "airbus_sas_not_detected",
            "same_ad_version_conflict",
        }
        review_flags = [
            flag
            for flag in record["review_flags"]
            if flag not in recomputed_flags
        ]

        if not ad_number:
            review_flags.append("ad_number_not_found")

        if ad_number_source == "pdf_header_filename_mismatch":
            review_flags.append(
                "ad_number_header_filename_mismatch"
            )

        if ad_number_source == "generic_first_page":
            review_flags.append(
                "generic_ad_number_requires_review"
            )

        if not is_airbus:
            review_flags.append("airbus_sas_not_detected")

        record["review_flags"] = sorted(set(review_flags))

    refreshed_ad_numbers = {
        record["file_instance_id"]: record["ad_number"]
        for record in records
    }
    for text_record in text_cache:
        text_record["ad_number"] = refreshed_ad_numbers.get(
            text_record["file_instance_id"],
            "",
        )

    print(
        "Using cached corpus for fast rerun:",
        len(records),
        "PDF records.",
    )
else:
    records = []
    text_cache = []
    normalized_text_by_file_id = {}

    for path in tqdm(pdf_paths, desc="Processing AD PDFs"):
        relative_path = path.relative_to(RAW_DIR)
        file_instance_id = stable_path_id(relative_path)

        stat = path.stat()
        file_hash = sha256_file(path)

        extraction_error = ""
        raw_text = ""
        pages = []
        page_count = 0
        pdf_metadata = {}

        try:
            pdf_result = extract_pdf_text(path)

            raw_text = pdf_result["text"]
            pages = pdf_result["pages"]
            page_count = pdf_result["page_count"]
            pdf_metadata = pdf_result["pdf_metadata"]

        except Exception as error:
            extraction_error = str(error)

        normalized_text = normalize_text(raw_text)
        normalized_text_hash = sha256_text(normalized_text)

        extracted_char_count = len(raw_text)
        average_chars_per_page = (
            extracted_char_count / max(page_count, 1)
        )

        needs_ocr = (
            extraction_error != ""
            or extracted_char_count < 300
            or average_chars_per_page < 100
        )

        if extraction_error:
            extraction_status = "failed"
        elif needs_ocr:
            extraction_status = "low_text"
        else:
            extraction_status = "ok"

        ad_number, ad_number_source, ad_number_confidence = find_ad_number(
            raw_text,
            path.name,
        )

        base_ad_number, revision_number, is_emergency = (
            parse_ad_components(ad_number)
        )

        (
            is_correction,
            correction_date_raw,
            correction_date,
        ) = extract_correction_information(raw_text)

        issue_date_raw, issue_date = extract_issue_date(raw_text)

        supersedes_ad_numbers, supersedure_evidence = (
            extract_supersedure_candidates(
                raw_text,
                ad_number,
            )
        )

        is_airbus_sas = bool(
            re.search(
                r"\bAirbus\b",
                raw_text,
                flags=re.IGNORECASE,
            )
        )

        review_flags = []

        if extraction_error:
            review_flags.append("text_extraction_failed")

        if needs_ocr:
            review_flags.append("needs_ocr")

        if not ad_number:
            review_flags.append("ad_number_not_found")

        if ad_number_source == "pdf_header_filename_mismatch":
            review_flags.append(
                "ad_number_header_filename_mismatch"
            )

        if ad_number_source == "generic_first_page":
            review_flags.append(
                "generic_ad_number_requires_review"
            )

        if not is_airbus_sas:
            review_flags.append("airbus_sas_not_detected")

        content_id = file_hash[:16]

        records.append({
            "file_instance_id": file_instance_id,
            "content_id": content_id,
            "file_name": path.name,
            "relative_path": str(relative_path),
            "file_size_bytes": stat.st_size,
            "modified_time_utc": datetime.fromtimestamp(
                stat.st_mtime,
                tz=timezone.utc,
            ).isoformat(),
            "page_count": page_count,
            "extracted_char_count": extracted_char_count,
            "average_chars_per_page": round(
                average_chars_per_page,
                2,
            ),
            "extraction_status": extraction_status,
            "needs_ocr": needs_ocr,
            "extraction_error": extraction_error,
            "pdf_title": pdf_metadata.get("title", ""),
            "pdf_author": pdf_metadata.get("author", ""),
            "file_sha256": file_hash,
            "normalized_text_sha256": normalized_text_hash,
            "ad_number": ad_number,
            "base_ad_number": base_ad_number,
            "revision_number": revision_number,
            "is_emergency": is_emergency,
            "is_correction": is_correction,
            "correction_date_raw": correction_date_raw,
            "correction_date": correction_date,
            "issue_date_raw": issue_date_raw,
            "issue_date": issue_date,
            "ad_number_source": ad_number_source,
            "ad_number_confidence": ad_number_confidence,
            "is_airbus_sas_detected": is_airbus_sas,
            "supersedes_ad_numbers": supersedes_ad_numbers,
            "supersedure_evidence": supersedure_evidence,
            "review_flags": review_flags,
        })

        text_cache.append({
            "file_instance_id": file_instance_id,
            "content_id": content_id,
            "relative_path": str(relative_path),
            "ad_number": ad_number,
            "text": raw_text,
        })

        normalized_text_by_file_id[file_instance_id] = normalized_text


Create the DataFrame:


In [ ]:
manifest = pd.DataFrame(records)

print("Manifest rows:", len(manifest))
display(manifest.head())


Validate the scan:


In [ ]:
assert len(manifest) == len(pdf_paths)
assert manifest["file_instance_id"].is_unique
assert manifest["file_sha256"].notna().all()

print("Initial scan completed successfully.")


# 10. Detect exact binary duplicates

An exact binary duplicate means:


```text
SHA-256(file A) == SHA-256(file B)
```

The files are byte-for-byte identical.

Run:


In [ ]:
def assign_duplicate_groups(
    dataframe,
    hash_column,
    group_column,
    prefix,
):
    dataframe[group_column] = ""

    valid_values = dataframe.loc[
        dataframe[hash_column].fillna("") != "",
        hash_column,
    ]

    counts = valid_values.value_counts()
    repeated_values = counts[counts > 1].index

    for hash_value in repeated_values:
        indexes = dataframe.index[
            dataframe[hash_column] == hash_value
        ].tolist()

        group_id = f"{prefix}-{hash_value[:12]}"

        dataframe.loc[indexes, group_column] = group_id


assign_duplicate_groups(
    manifest,
    hash_column="file_sha256",
    group_column="exact_binary_group",
    prefix="BIN",
)


Inspect detected groups:


In [ ]:
binary_duplicate_rows = manifest[
    manifest["exact_binary_group"] != ""
][[
    "exact_binary_group",
    "file_name",
    "relative_path",
    "file_size_bytes",
    "ad_number",
    "file_sha256",
]]

display(
    binary_duplicate_rows.sort_values(
        ["exact_binary_group", "relative_path"]
    )
)


# 11. Detect text-identical duplicates

Two PDFs can have different file hashes because of:

* Different PDF creation timestamps
* Different metadata
* Different compression
* Resaved PDF containers

However, their extracted text may be identical.

Run:


In [ ]:
assign_duplicate_groups(
    manifest,
    hash_column="normalized_text_sha256",
    group_column="exact_text_group",
    prefix="TXT",
)


Now assign a master file and duplicate type:


In [ ]:
manifest["duplicate_type"] = ""
manifest["duplicate_of"] = ""
manifest["safe_duplicate_candidate"] = False


def mark_duplicate_copies(
    dataframe,
    group_column,
    duplicate_type,
):
    grouped = dataframe[
        dataframe[group_column] != ""
    ].groupby(group_column)

    for group_id, group in grouped:
        # Deterministic master:
        # the lexicographically first relative path.
        sorted_group = group.sort_values("relative_path")

        master_index = sorted_group.index[0]
        master_id = dataframe.at[
            master_index,
            "file_instance_id",
        ]

        for duplicate_index in sorted_group.index[1:]:
            # Binary duplicates have the highest priority.
            if dataframe.at[duplicate_index, "duplicate_type"] == "":
                dataframe.at[
                    duplicate_index,
                    "duplicate_type",
                ] = duplicate_type

                dataframe.at[
                    duplicate_index,
                    "duplicate_of",
                ] = master_id

                dataframe.at[
                    duplicate_index,
                    "safe_duplicate_candidate",
                ] = True


mark_duplicate_copies(
    manifest,
    group_column="exact_binary_group",
    duplicate_type="exact_binary_duplicate",
)

mark_duplicate_copies(
    manifest,
    group_column="exact_text_group",
    duplicate_type="exact_text_duplicate",
)


Important: `safe_duplicate_candidate=True` does not mean “automatically delete.” It means the file is eligible for manual duplicate review.

# 12. Detect conflicting files for the same AD version

A revision is not a duplicate. These represent different versions:


```text
2020-0123
2020-0123R1
2020-0123R2
```

A correction can have the same AD number but a correction date:


```text
2020-0123R1
2020-0123R1 [Corrected: 12 August 2021]
```

Create a logical version key:


In [ ]:
def create_logical_version_key(row):
    if not row["ad_number"]:
        return f"UNPARSED|{row['file_instance_id']}"

    if row["is_correction"]:
        correction_identity = (
            row["correction_date"]
            or row["correction_date_raw"]
            or "unknown_date"
        )

        return (
            f"{row['ad_number']}"
            f"|CORRECTED:{correction_identity}"
        )

    return f"{row['ad_number']}|UNCORRECTED"


manifest["logical_version_key"] = manifest.apply(
    create_logical_version_key,
    axis=1,
)


Find cases where the same logical version has different content:


In [ ]:
for logical_key, group in manifest.groupby(
    "logical_version_key"
):
    if logical_key.startswith("UNPARSED"):
        continue

    # Ignore files already marked as exact duplicates.
    surviving_files = group[
        manifest.loc[group.index, "duplicate_of"] == ""
    ]

    if len(surviving_files) > 1:
        for index in surviving_files.index:
            if manifest.at[index, "duplicate_type"] == "":
                manifest.at[
                    index,
                    "duplicate_type",
                ] = "same_ad_version_different_content"

            flags = manifest.at[index, "review_flags"]

            if "same_ad_version_conflict" not in flags:
                flags.append("same_ad_version_conflict")


These files must not be removed automatically.

Possible causes include:

* A corrected PDF not detected by the regex
* One incomplete download
* Different publication copies
* Different language versions
* Incorrectly named files
* A PDF replaced by EASA without an obvious revision suffix

# 13. Construct revision and correction chains

Initialize the version fields:


In [ ]:
manifest["family_file_count"] = 0
manifest["family_version_count"] = 0
manifest["version_order"] = pd.NA
manifest["version_label"] = ""
manifest["previous_version"] = ""
manifest["next_version"] = ""
manifest["latest_version"] = ""
manifest["is_latest_version"] = False
manifest["has_revision_history"] = False


Create a readable version label:


In [ ]:
def create_version_label(row):
    if not row["ad_number"]:
        return ""

    label = row["ad_number"]

    if row["is_correction"]:
        correction_text = (
            row["correction_date"]
            or row["correction_date_raw"]
            or "date unknown"
        )

        label += f" [Corrected: {correction_text}]"

    return label


manifest["version_label"] = manifest.apply(
    create_version_label,
    axis=1,
)


Build the chains:


In [ ]:
valid_families = manifest[
    manifest["base_ad_number"] != ""
].groupby("base_ad_number")


for base_ad_number, family in valid_families:
    # Exclude exact duplicate copies when constructing the
    # logical sequence.
    logical_files = family[
        family["duplicate_of"] == ""
    ].copy()

    logical_files = logical_files.drop_duplicates(
        subset=["logical_version_key"],
        keep="first",
    )

    # Emergency versions occur before normal/revised versions.
    logical_files["_revision_rank"] = logical_files.apply(
        lambda row: (
            -1
            if row["is_emergency"]
            else int(row["revision_number"])
        ),
        axis=1,
    )

    logical_files["_correction_rank"] = (
        logical_files["is_correction"].astype(int)
    )

    logical_files["_date_rank"] = pd.to_datetime(
        logical_files["correction_date"].where(
            logical_files["is_correction"],
            logical_files["issue_date"],
        ),
        errors="coerce",
    )

    logical_files = logical_files.sort_values(
        by=[
            "_revision_rank",
            "_correction_rank",
            "_date_rank",
            "relative_path",
        ],
        na_position="last",
    )

    logical_keys = logical_files[
        "logical_version_key"
    ].tolist()

    labels = logical_files[
        "version_label"
    ].tolist()

    latest_label = labels[-1] if labels else ""

    family_file_count = len(family)
    family_version_count = len(logical_files)

    has_history = (
        family_version_count > 1
        or logical_files["revision_number"].max() > 0
        or logical_files["is_emergency"].any()
    )

    version_metadata = {}

    for position, logical_key in enumerate(logical_keys):
        previous_label = (
            labels[position - 1]
            if position > 0
            else ""
        )

        next_label = (
            labels[position + 1]
            if position < len(labels) - 1
            else ""
        )

        version_metadata[logical_key] = {
            "version_order": position + 1,
            "previous_version": previous_label,
            "next_version": next_label,
            "is_latest_version": (
                position == len(labels) - 1
            ),
        }

    # Apply version-chain data to every physical file,
    # including exact duplicate copies.
    for index in family.index:
        logical_key = manifest.at[
            index,
            "logical_version_key",
        ]

        metadata = version_metadata.get(logical_key, {})

        manifest.at[
            index,
            "family_file_count",
        ] = family_file_count

        manifest.at[
            index,
            "family_version_count",
        ] = family_version_count

        manifest.at[
            index,
            "has_revision_history",
        ] = has_history

        manifest.at[
            index,
            "latest_version",
        ] = latest_label

        if metadata:
            manifest.at[
                index,
                "version_order",
            ] = metadata["version_order"]

            manifest.at[
                index,
                "previous_version",
            ] = metadata["previous_version"]

            manifest.at[
                index,
                "next_version",
            ] = metadata["next_version"]

            manifest.at[
                index,
                "is_latest_version",
            ] = metadata["is_latest_version"]


Example expected chain:

| Base AD     | Version order | Version                      |
| ----------- | ------------: | ---------------------------- |
| `2020-0123` |             1 | `2020-0123`                  |
| `2020-0123` |             2 | `2020-0123R1`                |
| `2020-0123` |             3 | `2020-0123R1 [Corrected: …]` |
| `2020-0123` |             4 | `2020-0123R2`                |

# 14. Construct inverse supersedure links

If a document says:


```text
This AD supersedes EASA AD 2018-0123
```

then:


```text
Current AD → supersedes → 2018-0123
2018-0123 → superseded_by → Current AD
```

Run:


In [ ]:
superseded_by_map = defaultdict(set)
supersedure_link_rows = []

for _, row in manifest.iterrows():
    source_ad = row["ad_number"]

    if not source_ad:
        continue

    for target_ad in row["supersedes_ad_numbers"]:
        target_base, _, _ = parse_ad_components(target_ad)

        if not target_base:
            continue

        superseded_by_map[target_base].add(source_ad)

        supersedure_link_rows.append({
            "source_ad_number": source_ad,
            "source_file_instance_id": row["file_instance_id"],
            "target_ad_candidate": target_ad,
            "target_base_ad_number": target_base,
            "relationship": "supersedes_candidate",
            "evidence": " | ".join(
                row["supersedure_evidence"]
            ),
            "manually_verified": False,
        })


manifest["superseded_by_ad_numbers"] = (
    manifest["base_ad_number"].apply(
        lambda base_number: sorted(
            superseded_by_map.get(base_number, set())
        )
    )
)

supersedure_links = pd.DataFrame(
    supersedure_link_rows
)


Treat all of these as candidate relationships until manually checked.

# 15. Detect near duplicates

Near-duplicate detection identifies documents that are very similar but not identical.

This is useful for finding:

* Revisions
* Corrected issues
* Files saved with small text changes
* Reissued ADs with different numbers
* Incorrect filenames
* Incomplete copies

It must not be used for automatic deletion.

Run:


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors


Select files with usable text:


In [ ]:
eligible_manifest = manifest[
    (manifest["needs_ocr"] == False)
    & (manifest["extracted_char_count"] >= 1000)
].copy()

eligible_indexes = eligible_manifest.index.tolist()

near_duplicate_candidates = pd.DataFrame()

print(
    "Documents eligible for similarity analysis:",
    len(eligible_indexes),
)


Calculate similarities:


In [ ]:
if len(eligible_indexes) >= 2:
    similarity_texts = [
        normalized_text_by_file_id[
            manifest.at[index, "file_instance_id"]
        ]
        for index in eligible_indexes
    ]

    vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.98,
        max_features=60000,
        sublinear_tf=True,
        norm="l2",
    )

    text_matrix = vectorizer.fit_transform(
        similarity_texts
    )

    number_of_neighbors = min(
        8,
        len(eligible_indexes),
    )

    neighbor_model = NearestNeighbors(
        n_neighbors=number_of_neighbors,
        metric="cosine",
        algorithm="brute",
    )

    neighbor_model.fit(text_matrix)

    distances, neighbors = neighbor_model.kneighbors(
        text_matrix
    )

    NEAR_DUPLICATE_THRESHOLD = 0.92

    candidate_rows = []
    seen_pairs = set()

    for local_index, neighbor_list in enumerate(neighbors):
        original_index = eligible_indexes[local_index]

        for distance, neighbor_local_index in zip(
            distances[local_index],
            neighbor_list,
        ):
            neighbor_index = eligible_indexes[
                neighbor_local_index
            ]

            if original_index == neighbor_index:
                continue

            pair = tuple(
                sorted([original_index, neighbor_index])
            )

            if pair in seen_pairs:
                continue

            seen_pairs.add(pair)

            similarity = 1.0 - float(distance)

            if similarity < NEAR_DUPLICATE_THRESHOLD:
                continue

            row_a = manifest.loc[original_index]
            row_b = manifest.loc[neighbor_index]

            # Exact text duplicates are already handled.
            if (
                row_a["normalized_text_sha256"]
                == row_b["normalized_text_sha256"]
            ):
                continue

            if (
                row_a["base_ad_number"]
                and row_a["base_ad_number"]
                == row_b["base_ad_number"]
            ):
                relationship = (
                    "probable_revision_or_correction"
                )
            else:
                relationship = (
                    "possible_duplicate_or_reissued_ad"
                )

            candidate_rows.append({
                "file_instance_id_a": (
                    row_a["file_instance_id"]
                ),
                "file_instance_id_b": (
                    row_b["file_instance_id"]
                ),
                "ad_number_a": row_a["ad_number"],
                "ad_number_b": row_b["ad_number"],
                "relative_path_a": row_a["relative_path"],
                "relative_path_b": row_b["relative_path"],
                "text_similarity": round(
                    similarity,
                    5,
                ),
                "suggested_relationship": relationship,
                "manually_verified": False,
                "review_notes": "",
            })

    near_duplicate_candidates = pd.DataFrame(
        candidate_rows
    )

print(
    "Near-duplicate candidate pairs:",
    len(near_duplicate_candidates),
)


If too many unrelated candidates appear, raise the threshold:


In [ ]:
NEAR_DUPLICATE_THRESHOLD = 0.95


If known revisions are missed, lower it carefully:


In [ ]:
NEAR_DUPLICATE_THRESHOLD = 0.88


A reasonable starting value is `0.92`.

# 16. Add summary fields and perform validation

Run:


In [ ]:
manifest["requires_manual_review"] = manifest.apply(
    lambda row: (
        len(row["review_flags"]) > 0
        or row["duplicate_type"]
            == "same_ad_version_different_content"
    ),
    axis=1,
)


Calculate corpus statistics:


In [ ]:
summary = {
    "total_pdf_files": int(len(manifest)),
    "successfully_parsed_ad_numbers": int(
        manifest["ad_number"].ne("").sum()
    ),
    "unparsed_ad_numbers": int(
        manifest["ad_number"].eq("").sum()
    ),
    "unique_base_ad_numbers": int(
        manifest.loc[
            manifest["base_ad_number"] != "",
            "base_ad_number",
        ].nunique()
    ),
    "unique_logical_versions": int(
        manifest.loc[
            manifest["ad_number"] != "",
            "logical_version_key",
        ].nunique()
    ),
    "revised_ad_files": int(
        (manifest["revision_number"] > 0).sum()
    ),
    "correction_files": int(
        manifest["is_correction"].sum()
    ),
    "emergency_ad_files": int(
        manifest["is_emergency"].sum()
    ),
    "files_needing_ocr": int(
        manifest["needs_ocr"].sum()
    ),
    "airbus_sas_not_detected": int(
        (~manifest["is_airbus_sas_detected"]).sum()
    ),
    "exact_duplicate_copies": int(
        manifest["safe_duplicate_candidate"].sum()
    ),
    "same_version_content_conflicts": int(
        (
            manifest["duplicate_type"]
            == "same_ad_version_different_content"
        ).sum()
    ),
    "near_duplicate_candidate_pairs": int(
        len(near_duplicate_candidates)
    ),
    "supersedure_candidate_links": int(
        len(supersedure_links)
    ),
}

summary


Perform final assertions:


In [ ]:
assert len(manifest) == len(pdf_paths)
assert manifest["file_instance_id"].is_unique
assert manifest["file_sha256"].ne("").all()

print("Corpus manifest validation passed.")


# 17. Export the outputs

Lists must be converted into readable text for CSV and Excel:


In [ ]:
export_manifest = manifest.copy()

list_columns = [
    "supersedes_ad_numbers",
    "superseded_by_ad_numbers",
    "supersedure_evidence",
    "review_flags",
]

for column in list_columns:
    export_manifest[column] = export_manifest[
        column
    ].apply(
        lambda value: " | ".join(value)
        if isinstance(value, list)
        else value
    )


Save the master manifest:


In [ ]:
manifest_csv_path = (
    OUTPUT_DIR / "corpus_manifest.csv"
)

manifest_excel_path = (
    OUTPUT_DIR / "corpus_manifest.xlsx"
)

manifest_parquet_path = (
    OUTPUT_DIR / "corpus_manifest.parquet"
)

export_manifest.to_csv(
    manifest_csv_path,
    index=False,
)

export_manifest.to_excel(
    manifest_excel_path,
    index=False,
)

export_manifest.to_parquet(
    manifest_parquet_path,
    index=False,
)


Save the extracted-text cache:


In [ ]:
text_cache_df = pd.DataFrame(text_cache)

text_cache_path = (
    OUTPUT_DIR / "corpus_extracted_text.parquet"
)

text_cache_df.to_parquet(
    text_cache_path,
    index=False,
)


Do not put the full text in your CSV or Excel manifest. Keeping it in Parquet makes later retrieval processing much easier.

Save duplicate reports:


In [ ]:
duplicate_review = export_manifest[
    (export_manifest["exact_binary_group"] != "")
    | (export_manifest["exact_text_group"] != "")
    | (export_manifest["duplicate_type"] != "")
].copy()

duplicate_review.to_csv(
    OUTPUT_DIR / "duplicate_review.csv",
    index=False,
)


Save version chains:


In [ ]:
version_chain_columns = [
    "base_ad_number",
    "version_order",
    "ad_number",
    "version_label",
    "issue_date",
    "is_emergency",
    "is_correction",
    "correction_date",
    "previous_version",
    "next_version",
    "latest_version",
    "is_latest_version",
    "file_instance_id",
    "relative_path",
]

version_chains = export_manifest[
    export_manifest["base_ad_number"] != ""
][version_chain_columns].sort_values(
    ["base_ad_number", "version_order"]
)

version_chains.to_csv(
    OUTPUT_DIR / "version_chains.csv",
    index=False,
)


Save near-duplicate candidates:


In [ ]:
near_duplicate_candidates.to_csv(
    OUTPUT_DIR / "near_duplicate_candidates.csv",
    index=False,
)


Save supersedure candidates:


In [ ]:
supersedure_links.to_csv(
    OUTPUT_DIR / "supersedure_links.csv",
    index=False,
)


Save problem files:


In [ ]:
problem_files = export_manifest[
    export_manifest["requires_manual_review"]
].copy()

problem_files.to_csv(
    OUTPUT_DIR / "processing_and_metadata_review.csv",
    index=False,
)


Save the summary:


In [ ]:
with open(
    OUTPUT_DIR / "corpus_summary.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        summary,
        file,
        indent=2,
        ensure_ascii=False,
    )


Print the created files:


In [ ]:
print("Created:")
print(manifest_csv_path)
print(manifest_excel_path)
print(manifest_parquet_path)
print(text_cache_path)
print(OUTPUT_DIR / "duplicate_review.csv")
print(OUTPUT_DIR / "version_chains.csv")
print(OUTPUT_DIR / "near_duplicate_candidates.csv")
print(OUTPUT_DIR / "supersedure_links.csv")
print(OUTPUT_DIR / "processing_and_metadata_review.csv")
print(OUTPUT_DIR / "corpus_summary.json")


# 18. Manually review the results

Open `duplicate_review.csv` first.

## A. Exact binary duplicates

For every `exact_binary_group`:

1. Confirm the files have the same SHA-256.
2. Confirm they open correctly.
3. Keep one master file.
4. Mark other copies as verified duplicates.
5. Do not remove anything until your supervisor or your data-management procedure permits it.

## B. Exact text duplicates

For every `exact_text_group`:

1. Compare the filenames and paths.
2. Confirm the AD number is the same.
3. Check whether one PDF is marked as corrected.
4. Check the first and final pages visually.
5. If the text is identical but the metadata differs, treat one as the master.

## C. Same-version content conflicts

For `same_ad_version_different_content`:

1. Open both PDFs.
2. Check whether one contains `[Corrected: date]`.
3. Check the issue date.
4. Check page count.
5. Check whether one is truncated.
6. Check whether one is another language.
7. Never automatically classify one as a disposable duplicate.

## D. Revision chains

Open `version_chains.csv` and verify:


```text
Original → R1 → corrected R1 → R2
```

Check for:

* Missing base numbers
* Incorrect revision order
* Emergency ADs positioned before revised versions
* Corrections incorrectly treated as revisions
* Different revision PDFs with identical hashes

A missing `R1` does not necessarily mean your corpus is wrong. EASA may currently provide only the latest revision, or your original download may not include the historical version.

## E. Supersedure relationships

For every row in `supersedure_links.csv`:

1. Open the source PDF.
2. Search for the older AD number.
3. Read the complete sentence.
4. Confirm that the relationship is actually “supersedes.”
5. Set `manually_verified=True`.
6. Record any uncertainty in a review-notes column.

## F. OCR problems

Open `processing_and_metadata_review.csv`.

For each `needs_ocr` file:

1. Open the PDF visually.
2. Determine whether it is scanned.
3. Check whether it is damaged or password protected.
4. Confirm whether the first page contains a readable AD number.
5. Record the correct AD number manually if necessary.

Do not exclude scanned documents at this stage. They may represent important older ADs.

# 19. Do not edit the generated manifest directly

Create a separate manual correction file:


In [ ]:
manual_override_columns = [
    "file_instance_id",
    "confirmed_ad_number",
    "confirmed_is_correction",
    "confirmed_correction_date",
    "confirmed_supersedes_ad_numbers",
    "duplicate_decision",
    "reviewer_name",
    "review_date",
    "review_notes",
]

manual_overrides_path = (
    OUTPUT_DIR / "manual_overrides.csv"
)

if not manual_overrides_path.exists():
    pd.DataFrame(
        columns=manual_override_columns
    ).to_csv(
        manual_overrides_path,
        index=False,
    )

print(manual_overrides_path)


Record manual corrections there rather than changing `corpus_manifest.csv`. This preserves reproducibility: the generated manifest can always be rebuilt, while human decisions remain in a separate audit file.

# Completion criteria

Step 1 is complete when:

* Every physical PDF has exactly one manifest row.
* Every PDF has a SHA-256 hash.
* At least 95% of the files have a parsed AD number, or all failures have been manually reviewed.
* Exact binary and text duplicates are grouped.
* Same-version content conflicts are manually reviewed.
* Revisions are grouped by `base_ad_number`.
* Corrections are distinguished from revisions.
* The latest available version is identified for every version family.
* Supersedure candidates have evidence snippets.
* OCR failures and non-Airbus detections are documented.
* No original PDF has been deleted or modified.
* `manual_overrides.csv` records every human correction.

The most important deliverables are `corpus_manifest.xlsx`, `version_chains.csv`, `duplicate_review.csv`, and `corpus_extracted_text.parquet`.
